# Arquitetura Básica da Web

## Conceitos-chave
- A web é estruturada em documentos HTML organizados em uma árvore chamada **DOM (Document Object Model)**.  
- Cada elemento da página (títulos, parágrafos, links, imagens, tabelas) é representado por **tags** e pode conter atributos como `id` e `class`.  
- Para coletar informações, usamos **seletores** (CSS ou XPath), que permitem identificar elementos de interesse na página.  
- Para engenheiros de dados, compreender o DOM é essencial para criar pipelines de scraping **reprodutíveis e escaláveis**, pois evita dependência de mudanças superficiais no layout.  

## 1.2 STATUS HTTP

<center><img src="./request.png"></img></center>


LEIA MAIS:<br>
> - CÓDIGOS HTTP: https://www.httpstatus.com.br
> - DOCS MOZILLA: https://developer.mozilla.org/pt-BR/docs/Web/HTTP/Basics_of_HTTP

## Estrutura de páginas web

## Contexto em Engenharia de Dados
- Seletores mal escolhidos (ex.: baseados em estilos visuais) tornam o pipeline instável.  
- Seletores bem planejados (ex.: atributos estáveis como `data-*`) reduzem manutenção.  
- Em ambientes de produção, a definição de seletores é comparável a um **contrato de schema**: se mudar, o pipeline quebra.  
- Uso prático: estruturar regras de extração de tabelas, listas de produtos, manchetes de portais e dados abertos.  

## Boas práticas de extração
- Sempre inspecionar a página com as ferramentas do navegador (DevTools).  
- Identificar **padrões repetidos** (listas, tabelas, cartões de produto).  
- Validar a extração em múltiplas páginas (não apenas no primeiro resultado).  
- Documentar seletores para facilitar manutenção.  

## Bibliotecas utilizadas
- **requests** → baixar HTML.  
- **BeautifulSoup (bs4)** → navegar e extrair dados via seletores CSS.  
- **lxml** → parser rápido para HTML/XML.  
- **pandas** → estruturar os dados em DataFrames.  

--------------------------------------------------------------------------------
### 🐍 Lab 1 — Extraindo manchetes do G1

Objetivo:  
Coletar os títulos de manchetes do G1 CE, extraindo texto e links diretamente do HTML.

In [1]:
import os
import uuid
import logging
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
URL = "https://g1.globo.com/ce/ceara/" 
OUT =  "g1_ceara.parquet"
LOG_FILE = "g1_ceara_scraper.log"
UA = "Mozilla/5.0 (compatible; DataEngScraper/1.0)"
TIMEOUT = 15

In [3]:
# Logger básico
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.FileHandler(LOG_FILE, "a", "utf-8"), logging.StreamHandler()]
)
logger = logging.getLogger("g1_log")

In [4]:
try:
    logger.info(f"Acessando {URL}")
    resp = requests.get(URL, headers={"User-Agent": UA}, timeout=TIMEOUT)
    resp.raise_for_status()
except Exception as e:
    logger.exception(f"Erro ao baixar {URL}: {e}")
    raise

2025-10-03 14:39:57,952 | INFO | g1_log | Acessando https://g1.globo.com/ce/ceara/


In [8]:
soup = BeautifulSoup(resp.text, "html.parser")
seen = set()
rows = []

In [11]:
for post in soup.select("div.feed-post"):
    a = post.select_one("a.feed-post-link")
    if not a:
        continue
    date = post.select_one("span.feed-post-datetime")
    title, link = a.get_text(strip=True), a.get("href")
    
    if not title or not link or link in seen: 
        continue
    
    seen.add(link)
    
    rows.append({"id": str(uuid.uuid4()), 
                 "title": title,
                 "link": link, 
                 "published_at": date.get_text(strip=True) if date else None})

df = pd.DataFrame(rows)
df["extracted_at"] = pd.Timestamp.utcnow()

logger.info(f"Total de manchetes extraídas: {len(df)}")
if not df.empty:
    df.to_parquet(OUT, index=False)
    logger.info(f"Parquet salvo em {os.path.abspath(OUT)}")
else:
    logger.warning("Nenhuma manchete encontrada.")


2025-10-03 14:40:02,698 | INFO | g1_log | Total de manchetes extraídas: 7
2025-10-03 14:40:02,714 | INFO | g1_log | Parquet salvo em C:\Users\ProDigital\Machine\UNIFOR\MBA ENG.DADOS\T2\Web Mining\Aula 02\g1_ceara.parquet


In [13]:
df

,id,title,link,published_at,extracted_at
0,d234d05b-dfec-4fd7-928d-cb2ebed6d12e,Itamaraty pede libertação de Luizianne e outro...,https://g1.globo.com/ce/ceara/noticia/2025/10/...,Há 6 horas,2025-10-03 17:40:02.697831+00:00
1,cbac6fbe-4c80-4dea-84aa-6b2cec6d8a29,"Internada em hospital, grávida ganha ensaio fo...",https://g1.globo.com/ce/ceara/noticia/2025/10/...,Há 9 horas,2025-10-03 17:40:02.697831+00:00
2,386dcff9-a5ad-4921-beed-d0d77dc54d97,Dupla suspeita de homicídio tenta fugir mas ba...,https://g1.globo.com/ce/ceara/noticia/2025/10/...,Há 18 horas,2025-10-03 17:40:02.697831+00:00
3,11ae3af3-5aee-46c5-8a79-25d9f8ea8835,Policial que publicou vídeo lavando viatura é ...,https://g1.globo.com/ce/ceara/noticia/2025/10/...,Ontem,2025-10-03 17:40:02.697831+00:00
4,3443bcaa-d84f-4969-842b-7c6fb99a1f32,Morrinhos abre concurso com 419 vagas e salári...,https://g1.globo.com/ce/ceara/suachance/notici...,Há 4 horas,2025-10-03 17:40:02.697831+00:00
5,2574f6bc-c14d-4a23-af9d-e415157182e4,Fortaleza realiza Dia D de vacinação antirrábi...,https://g1.globo.com/ce/ceara/noticia/2025/10/...,Há 8 horas,2025-10-03 17:40:02.697831+00:00
6,4467b7cd-d621-4f75-b0af-ec29bd3acd76,Navios da Marinha usados para pesquisa abrem p...,https://g1.globo.com/ce/ceara/noticia/2025/10/...,Há 8 horas,2025-10-03 17:40:02.697831+00:00


In [ ]:
/html/body/div[2]/main/div[5]/div[2]/div/div/div/div/div/div/div/div[2]/div/div/div/div[1]/div/div/div/div[2]/div/h2/a/p

In [ ]:
//*[@id="ab720b1c-69af-43b1-b3bd-af7b31449af8"]/div/div[2]/div/h2/a/p

In [ ]:
#df.assign(extracted_at=pd.Timestamp.utcnow()).to_parquet(OUT_PARQUET, index=False)